In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

from torchvision import datasets, transforms

In [19]:
class Net(nn.Module):

    def __init__(self, ):
        super().__init__()
        self.rnn = nn.LSTM(input_size=28, hidden_size=128, num_layers=2,
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.batchnorm = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, input):
        output = input.reshape(-1, 28, 28)
        output, hidden = self.rnn(output)
        output = output[:, -1, :]
        output = self.batchnorm(output)
        output = self.dropout(output)
        output = self.fc1(output)
        output = F.relu(output)
        output = self.dropout(output)
        output = self.fc2(output)
        output = F.log_softmax(output, dim=1)
        return output

In [11]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

In [21]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [24]:
class CNN_LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            32, 64, kernel_size=3, padding=1
        )

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        self.lstm = nn.LSTM(
            input_size=64 * 14,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(x.size(0), 14, 64 * 14)

        output, hidden = self.lstm(x)

        x = output[:, -1, :]
        x = self.dropout(x)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [ ]:
use_cuda = torch.cuda.is_available()
torch.manual_seed(45123)

device = torch.device('cuda' if use_cuda else 'cpu')

kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=True, download=True,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081))
                   ])),
    batch_size=64, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=False,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081))])),
    batch_size=1000, shuffle=True, **kwargs)

model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=0.1)

scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

for epoch in range(1, 20):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), 'MNIST_RNN.pt')

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.353675
Train Epoch: 1 [640/60000 (1%)]	Loss: 2.332362
Train Epoch: 1 [1280/60000 (2%)]	Loss: 2.320247
Train Epoch: 1 [1920/60000 (3%)]	Loss: 2.143547
Train Epoch: 1 [2560/60000 (4%)]	Loss: 2.087197
Train Epoch: 1 [3200/60000 (5%)]	Loss: 2.062738
Train Epoch: 1 [3840/60000 (6%)]	Loss: 2.012636
Train Epoch: 1 [4480/60000 (7%)]	Loss: 2.099581
Train Epoch: 1 [5120/60000 (9%)]	Loss: 1.955158
Train Epoch: 1 [5760/60000 (10%)]	Loss: 1.831061
Train Epoch: 1 [6400/60000 (11%)]	Loss: 1.873747
Train Epoch: 1 [7040/60000 (12%)]	Loss: 1.765295
Train Epoch: 1 [7680/60000 (13%)]	Loss: 1.595481
Train Epoch: 1 [8320/60000 (14%)]	Loss: 1.620371
Train Epoch: 1 [8960/60000 (15%)]	Loss: 1.952485
Train Epoch: 1 [9600/60000 (16%)]	Loss: 1.694388
Train Epoch: 1 [10240/60000 (17%)]	Loss: 1.721907
Train Epoch: 1 [10880/60000 (18%)]	Loss: 1.514125
Train Epoch: 1 [11520/60000 (19%)]	Loss: 1.574369
Train Epoch: 1 [12160/60000 (20%)]	Loss: 1.459792
Train Epoch: 1 [12800/60000 (

In [25]:
model = CNN_LSTM().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=0.1)

scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

for epoch in range(1, 20):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), 'MNIST_RNN_CNN.pt')

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.310631
Train Epoch: 1 [640/60000 (1%)]	Loss: 2.298762
Train Epoch: 1 [1280/60000 (2%)]	Loss: 2.307488
Train Epoch: 1 [1920/60000 (3%)]	Loss: 2.312692
Train Epoch: 1 [2560/60000 (4%)]	Loss: 2.290563
Train Epoch: 1 [3200/60000 (5%)]	Loss: 2.311294
Train Epoch: 1 [3840/60000 (6%)]	Loss: 2.304080
Train Epoch: 1 [4480/60000 (7%)]	Loss: 2.300796
Train Epoch: 1 [5120/60000 (9%)]	Loss: 2.312878
Train Epoch: 1 [5760/60000 (10%)]	Loss: 2.299320
Train Epoch: 1 [6400/60000 (11%)]	Loss: 2.298626
Train Epoch: 1 [7040/60000 (12%)]	Loss: 2.303078
Train Epoch: 1 [7680/60000 (13%)]	Loss: 2.298105
Train Epoch: 1 [8320/60000 (14%)]	Loss: 2.298266
Train Epoch: 1 [8960/60000 (15%)]	Loss: 2.281335
Train Epoch: 1 [9600/60000 (16%)]	Loss: 2.283630
Train Epoch: 1 [10240/60000 (17%)]	Loss: 2.281224
Train Epoch: 1 [10880/60000 (18%)]	Loss: 2.275311
Train Epoch: 1 [11520/60000 (19%)]	Loss: 2.262279
Train Epoch: 1 [12160/60000 (20%)]	Loss: 2.254733
Train Epoch: 1 [12800/60000 (